# Cox Proportional Hazard Modeling
Beginning from the most simple clinical metadata, we will test Cox Proportional Hazards Modeling to predict cancer survival

In [1]:
import pandas as pd
from lifelines import CoxPHFitter
import pyhere as here

In [2]:
here.here()

PosixPath('/Users/jmakings/Documents/Projects/breast_cancer_survival_prediction')

In [3]:
# load clinical metadata with PCA + UMAP features
pca_clinical_df = pd.read_csv(here.here("data", "processed","clinical_dimred_features.csv"))

In [4]:
pca_clinical_df

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,PC293_harmony,PC294_harmony,PC295_harmony,PC296_harmony,PC297_harmony,PC298_harmony,PC299_harmony,PC300_harmony,UMAP1,UMAP2
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0.244572,-0.496506,0.168578,0.533063,0.434756,0.272053,0.324875,-0.103091,-1.800003,-6.149750
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0.303471,-0.844191,-0.596785,0.820834,0.064187,-0.056478,0.407295,0.292778,-2.570254,-2.240858
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0.059991,-0.518747,0.594953,0.316483,0.071531,0.090688,0.333440,0.199574,-5.767281,-4.865173
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,-0.689182,0.091264,-0.301379,-0.521918,0.213592,-0.737401,1.100436,-0.762680,-5.721486,-4.890594
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,-0.590699,-0.293891,1.523677,-1.229448,0.036228,0.395598,0.133042,0.194311,-1.569188,-1.622840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,7295,43.10,BREAST CONSERVING,Breast Cancer,Breast Invasive Lobular Carcinoma,High,0,LumA,4.0,Positve,...,-0.457317,0.608927,0.001513,-0.051347,0.044357,-0.138593,-0.374052,0.445042,-1.645502,-3.813061
1900,7296,42.88,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,...,0.176880,0.559336,0.212478,0.075198,-0.336113,-0.213558,0.347952,0.099582,-1.342959,-1.388248
1901,7297,62.90,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,...,-0.460358,0.201798,-0.437061,0.109682,-0.401253,-0.174120,-0.289600,0.339712,-3.293207,-0.444603
1902,7298,61.16,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,Moderate,0,LumB,4.0,Positve,...,0.399676,-0.208885,0.881920,0.048649,0.519081,-0.593877,-0.908927,0.127365,-3.406587,-0.689774


### Preprocess data for cox model (One-hot encoding)

In [13]:
# One hot encode categorical variables

# Identify categorical columns automatically
cat_cols = pca_clinical_df.select_dtypes(include=["object", "category"]).columns

# Apply one-hot encoding only to categorical columns
df_encoded = pd.get_dummies(pca_clinical_df, columns=cat_cols, drop_first=True)

# Change boolean columns to integers
bool_cols = df_encoded.select_dtypes(include=["bool"]).columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)


In [16]:
pca_clinical_df

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,...,PC293_harmony,PC294_harmony,PC295_harmony,PC296_harmony,PC297_harmony,PC298_harmony,PC299_harmony,PC300_harmony,UMAP1,UMAP2
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,...,0.244572,-0.496506,0.168578,0.533063,0.434756,0.272053,0.324875,-0.103091,-1.800003,-6.149750
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,...,0.303471,-0.844191,-0.596785,0.820834,0.064187,-0.056478,0.407295,0.292778,-2.570254,-2.240858
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,...,0.059991,-0.518747,0.594953,0.316483,0.071531,0.090688,0.333440,0.199574,-5.767281,-4.865173
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,...,-0.689182,0.091264,-0.301379,-0.521918,0.213592,-0.737401,1.100436,-0.762680,-5.721486,-4.890594
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,...,-0.590699,-0.293891,1.523677,-1.229448,0.036228,0.395598,0.133042,0.194311,-1.569188,-1.622840
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1899,7295,43.10,BREAST CONSERVING,Breast Cancer,Breast Invasive Lobular Carcinoma,High,0,LumA,4.0,Positve,...,-0.457317,0.608927,0.001513,-0.051347,0.044357,-0.138593,-0.374052,0.445042,-1.645502,-3.813061
1900,7296,42.88,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,...,0.176880,0.559336,0.212478,0.075198,-0.336113,-0.213558,0.347952,0.099582,-1.342959,-1.388248
1901,7297,62.90,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumB,4.0,Positve,...,-0.460358,0.201798,-0.437061,0.109682,-0.401253,-0.174120,-0.289600,0.339712,-3.293207,-0.444603
1902,7298,61.16,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,Moderate,0,LumB,4.0,Positve,...,0.399676,-0.208885,0.881920,0.048649,0.519081,-0.593877,-0.908927,0.127365,-3.406587,-0.689774


In [18]:
# save one-hot encoded clinical metadata with PCA + UMAP features
df_encoded.to_csv(here.here("data", "processed","clinical_dimred_features_onehot.csv"), index=False)

In [ ]:
# fit Cox Proportional Hazards model
cph = CoxPHFitter()
